# Follow-ups on the drug reviews run

Two follow-up analyses on top of `quickstart_drug_reviews.ipynb` (same data sample, split, embeddings and SAE settings):

- **Part A: re-interpreting the most predictive neuron.** In the main run, the neuron most correlated with the label (r = 0.66) was labeled "mentions a specific dosage in milligrams". That label had poor fidelity (F1 = 0.29) and the hypothesis did not predict held-out ratings (p = 0.94). Because the SAE is retrained here, neuron numbers change, so this tests the same idea on the most predictive neuron of the new SAE: we generate 3 candidate labels for it, once with the task instructions from the main run (which ask for specific content rather than general satisfaction) and once with neutral instructions, and check whether a better label recovers the neuron's predictive signal.
- **Part B: embedding similarity vs. LLM annotation.** Annotating 1,200 held-out reviews × 20 hypotheses with the LLM takes ~30 minutes. Here we compare it against a much cheaper alternative: cosine similarity between each review's embedding and each hypothesis's embedding. How well do the two agree, and do they lead to the same conclusions about which hypotheses are predictive?

Needs the `MODAL_VLLM_KEY` Colab secret. A T4 GPU runtime is faster, but a CPU runtime also works: the LLM runs on Modal, and only the embeddings and the small SAE run in Colab (timings in Part B then reflect CPU embedding).

In [1]:
!pip install -q git+https://github.com/psnandini/HypotheSAEs.git
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU: running on CPU'


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
import os
from google.colab import userdata, files

os.environ["OPENAI_BASE_URL"] = "https://psnandini--hypothesaes-vllm-serve.us-east.modal.run/v1"
os.environ["OPENAI_KEY_SAE"] = userdata.get("MODAL_VLLM_KEY")
os.environ["EMB_CACHE_DIR"] = "/content/emb_cache"

import html, time
import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr
from sklearn.metrics import roc_auc_score
from sentence_transformers import SentenceTransformer

from hypothesaes.quickstart import train_sae
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.llm_api import get_completion
from hypothesaes.interpret_neurons import NeuronInterpreter, InterpretConfig, ScoringConfig, LLMConfig, SamplingConfig
from hypothesaes.annotate import annotate_texts_with_concepts
from hypothesaes.evaluation import score_hypotheses, compute_ols_metrics

MODEL = "Qwen/Qwen3-8B"
ANNOTATION_KWARGS = {"prompt_suffix": " /no_think"}   # thinking off for yes/no annotation
OUT = "/content/results"
os.makedirs(OUT, exist_ok=True)
pd.set_option("display.max_colwidth", None)

t = time.time()
print(get_completion(prompt="Reply with the single word: ready /no_think", model=MODEL), f"({time.time() - t:.0f} s)")



ready (62 s)


## Same data, split, embeddings and SAE as the main run

This cell repeats the main notebook's sampling (fixed random seeds), so the train / held-out reviews are identical.

In [3]:
!wget -q -nc https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip
!unzip -q -o drugsCom_raw.zip -d drugs

df = pd.concat([pd.read_csv(f"drugs/{f}", sep="\t") for f in ["drugsComTrain_raw.tsv", "drugsComTest_raw.tsv"]])
df["review"] = df.review.map(html.unescape).str.strip().str.strip('"').str.strip()
df = df.drop_duplicates(subset="review")
df = df[(df.rating >= 9) | (df.rating <= 4)].copy()
df["label"] = (df.rating >= 9).astype(int)
df = df.groupby("label").sample(n=3000, random_state=0).sample(frac=1, random_state=0).reset_index(drop=True)
n = len(df)
df["split"] = np.select([df.index < 0.7 * n, df.index < 0.8 * n], ["train", "val"], "holdout")
train_df, val_df, holdout_df = (df[df.split == s].reset_index(drop=True) for s in ["train", "val", "holdout"])
texts, labels = train_df.review.tolist(), train_df.label.values
holdout_texts, holdout_labels = holdout_df.review.tolist(), holdout_df.label.values
print(len(texts), len(val_df), len(holdout_texts))

EMBEDDER = "nomic-ai/modernbert-embed-base"
text2embedding = get_local_embeddings(texts + val_df.review.tolist(), model=EMBEDDER, batch_size=64, cache_name="drugs_modernbert")
train_embeddings = np.stack([text2embedding[t] for t in texts])
val_embeddings = np.stack([text2embedding[t] for t in val_df.review])

# Hypotheses from the main run
results = pd.read_csv("https://raw.githubusercontent.com/psnandini/HypotheSAEs/main/results/drugs_hypotheses.csv")
hypotheses = results.interpretation.tolist()

4200 600 1200


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/445k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

... (7 outputs left out)

Loaded model nomic-ai/modernbert-embed-base to cuda


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/75 [00:00<?, ?it/s]

Saved 4800 embeddings to /content/emb_cache/drugs_modernbert/chunk_000.npy


SAE: a new Colab notebook runs on a fresh machine, so the main run's SAE checkpoint is not available here and the SAE is retrained with the same settings (now with a fixed seed, so this notebook is reproducible). Neuron numbers therefore differ from the main run. Part A uses the neuron most correlated with the label in this SAE; its correlation is printed for comparison with the main run's top neuron (r = 0.66). The main run's label for that neuron ("mentions a specific dosage in milligrams") is a plain-text hypothesis, so it is tested on held-out data regardless.

In [4]:
torch.manual_seed(0); np.random.seed(0)
sae = train_sae(embeddings=train_embeddings, M=256, K=8, matryoshka_prefix_lengths=[32, 256],
                checkpoint_dir="/content/checkpoints/drugs_followup", val_embeddings=val_embeddings)

activations = sae.get_activations(torch.tensor(train_embeddings, dtype=torch.float))
corrs = np.array([pearsonr(activations[:, i], labels)[0] if activations[:, i].std() > 0 else 0 for i in range(activations.shape[1])])
NEURON = int(np.nanargmax(np.abs(corrs)))
print(f"Neuron {NEURON}: correlation with label {corrs[NEURON]:+.3f}, active in {(activations[:, NEURON] > 0).mean():.1%} of training reviews")

  0%|          | 0/100 [00:00<?, ?it/s]

Saved model to /content/checkpoints/drugs_followup/SAE_matryoshka_M=256_K=8_prefixes=32-256.pt


Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Neuron 11: correlation with label -0.404, active in 21.1% of training reviews


## Part A: what does the most predictive neuron actually encode?

First, look at the reviews it fires on most strongly, and compare active vs. inactive reviews on simple properties.

In [5]:
act = activations[:, NEURON]
active = act > 0
n_words = np.array([len(t.split()) for t in texts])
print(f"positive-review rate: active {labels[active].mean():.2f} vs inactive {labels[~active].mean():.2f}")
print(f"median words:         active {np.median(n_words[active]):.0f} vs inactive {np.median(n_words[~active]):.0f}")
print(f"mentions 'mg':        active {np.mean(['mg' in t.lower() for t in np.array(texts)[active]]):.2f} vs inactive {np.mean(['mg' in t.lower() for t in np.array(texts)[~active]]):.2f}")
print()
for rank, i in enumerate(np.argsort(act)[::-1][:8], 1):
    print(f"{rank}. [{'positive' if labels[i] else 'negative'}] {texts[i][:300]}")
    print()

positive-review rate: active 0.09 vs inactive 0.61
median words:         active 69 vs inactive 84


mentions 'mg':        active 0.16 vs inactive 0.20

1. [negative] They gave me this medication via IV in the ER for a very intense migraine and extreme nausea, I would never recommend this medication to anyone ever, the second it was pushed through, I started having an extreme panic attack, something like I have never felt before, like a huge wave of doom rushed o

2. [negative] I was given samples of the 25mg dose for constipation due to methadone.  I took it as prescribed, 2 hours after a meal.  Within an hour, I was vomiting, has severe diarrhea, and horrible stomach cramps.  I ended up calling an ambulance and spending 2 days in the hospital.  All my electrolytes were o

3. [negative] This drug give me hallucinations And made my body tingle.

4. [negative] Still felt nauseous after being given this drug after surgery. I woke up extremely sick and vomiting. This was the third drug given to help combat these effects. I didn't know I'd been given it but my heart started racing and con

Generate 3 candidate labels under each instruction set, then score every candidate's fidelity on 100 training reviews (50 where the neuron fires most, 50 where it is silent).

In [6]:
ORIGINAL_INSTRUCTIONS = """All of the texts are patient reviews of a medication, posted on Drugs.com.
Features should describe a specific property of the review's content, such as a particular symptom, side effect, or experience,
rather than general satisfaction with the drug. For example:
- "mentions a specific dosage in milligrams"
- "mentions taking the medication for several years"
"""
NEUTRAL_INSTRUCTIONS = """All of the texts are patient reviews of a medication, posted on Drugs.com.
Features should describe a specific property of the review's content or writing."""

interpreter = NeuronInterpreter(interpreter_model=MODEL, annotator_model=MODEL,
                                n_workers_interpretation=8, n_workers_annotation=32)
candidates = []
for name, instructions in [("original", ORIGINAL_INSTRUCTIONS), ("neutral", NEUTRAL_INSTRUCTIONS)]:
    config = InterpretConfig(sampling=SamplingConfig(n_examples=20, max_words_per_example=256),
                             llm=LLMConfig(temperature=0.7), n_candidates=3,
                             task_specific_instructions=instructions)
    interps = interpreter.interpret_neurons(texts=texts, activations=activations, neuron_indices=[NEURON], config=config)
    candidates += [(name, c) for c in interps[NEURON] if c]

scores = interpreter.score_interpretations(texts=texts, activations=activations,
                                           interpretations={NEURON: [c for _, c in candidates]},
                                           config=ScoringConfig(n_examples=100), **ANNOTATION_KWARGS)
cand_df = pd.DataFrame([{"instructions": name, "interpretation": c, **scores[NEURON][c]} for name, c in candidates])
cand_df = cand_df.sort_values("f1", ascending=False).reset_index(drop=True)
display(cand_df)

Generating interpretations:   0%|          | 0/3 [00:00<?, ?it/s]

Generating interpretations:   0%|          | 0/3 [00:00<?, ?it/s]

Found 0 cached items; annotating 600 uncached items


Scoring neuron interpretation fidelity (1 neurons; 6 candidate interps per neuron; 100 examples to score each …

,instructions,interpretation,recall,precision,f1,correlation
0,original,"mentions specific severe side effects such as hallucinations, vomiting, or respiratory distress",0.86,0.767857,0.811321,0.598093
1,original,"mentions severe adverse effects such as respiratory distress, hallucinations, or suicidal thoughts",0.84,0.777778,0.807692,0.598327
2,neutral,describes severe adverse effects requiring medical intervention,0.90,0.652174,0.756303,0.459269
3,neutral,"mentions severe adverse effects including vomiting, hallucinations, and respiratory distress",0.74,0.770833,0.755102,0.522508
4,neutral,"mentions severe adverse effects requiring hospitalization or life-threatening symptoms such as respiratory distress, electrolyte imbalance, and suicidal thoughts",0.76,0.730769,0.745098,0.480591
5,original,mentions severe adverse effects requiring hospitalization or leading to life-threatening conditions,0.76,0.716981,0.737864,0.461233


Now test on held-out reviews: does the best new label predict ratings better than the original "milligrams" label? (Each label is tested on its own, so the numbers are not affected by the other 19 hypotheses.)

In [7]:
old_label = results.loc[results.neuron_idx == 5, "interpretation"].item()   # main run's top neuron
test_labels = [old_label, cand_df.interpretation[0]]
ann = annotate_texts_with_concepts(texts=holdout_texts, concepts=test_labels, model=MODEL, cache_name="drugs_followup",
                                   max_words_per_example=256, n_workers=32, **ANNOTATION_KWARGS)
rows = []
for h in test_labels:
    m, table = score_hypotheses({h: ann[h]}, holdout_labels, classification=True)
    rows.append({"label": h, "source": "main run label" if h == old_label else "best new candidate",
                 "held-out AUROC": m["auroc"], **table.iloc[0].drop("hypothesis").to_dict()})
partA = pd.DataFrame(rows)
display(partA)
cand_df.to_csv(f"{OUT}/followup_neuron_candidates.csv", index=False)
partA.to_csv(f"{OUT}/followup_neuron_holdout.csv", index=False)
files.download(f"{OUT}/followup_neuron_candidates.csv")
files.download(f"{OUT}/followup_neuron_holdout.csv")

Found 0 cached items; annotating 2400 uncached items


Annotating:   0%|          | 0/2400 [00:00<?, ?it/s]

Optimization terminated successfully.
         Current function value: 0.691244
         Iterations 4
Optimization terminated successfully.
         Current function value: 0.572185
         Iterations 5


,label,source,held-out AUROC,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence
0,mentions a specific dosage in milligrams,main run label,0.526594,0.069541,3.515716e-02,0.279152,3.539079e-02,0.257500
1,"mentions specific severe side effects such as hallucinations, vomiting, or respiratory distress",best new candidate,0.739541,-0.483552,1.334455e-70,-2.114900,7.444214e-57,0.451667


(saved /content/results/followup_neuron_candidates.csv)
(saved /content/results/followup_neuron_holdout.csv)


## Part B: embedding similarity vs. LLM annotation

**LLM annotation** (as in the main run): Qwen3-8B answers yes/no for each of the 1,200 held-out reviews × 20 hypotheses. This is the slow step (~30 min); the answers are saved right away.

In [8]:
t = time.time()
llm_ann = annotate_texts_with_concepts(texts=holdout_texts, concepts=hypotheses, model=MODEL, cache_name="drugs_followup",
                                       max_words_per_example=256, n_workers=32, **ANNOTATION_KWARGS)
llm_seconds = time.time() - t
llm_df = pd.DataFrame(llm_ann)
llm_df.to_csv(f"{OUT}/followup_llm_annotations.csv", index=False)
files.download(f"{OUT}/followup_llm_annotations.csv")
print(f"LLM annotation: {llm_seconds / 60:.1f} min")

Found 1200 cached items; annotating 22800 uncached items


Annotating:   0%|          | 0/22800 [00:00<?, ?it/s]

(saved /content/results/followup_llm_annotations.csv)
LLM annotation: 31.3 min


**Embedding similarity:** embed each review and each hypothesis with the same embedding model, and score each (review, hypothesis) pair by cosine similarity. `modernbert-embed-base` was trained for retrieval with `search_query:` / `search_document:` prefixes, so the hypothesis is treated as the query.

In [9]:
t = time.time()
encoder = SentenceTransformer(EMBEDDER, device="cuda" if torch.cuda.is_available() else "cpu")
doc_emb = encoder.encode(["search_document: " + x for x in holdout_texts], batch_size=64, normalize_embeddings=True)
hyp_emb = encoder.encode(["search_query: " + h for h in hypotheses], normalize_embeddings=True)
cos = doc_emb @ hyp_emb.T            # (1200 reviews, 20 hypotheses)
emb_seconds = time.time() - t
print(f"Embedding similarity: {emb_seconds:.0f} s  vs  LLM annotation: {llm_seconds:.0f} s")

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Embedding similarity: 13 s  vs  LLM annotation: 1877 s


**1. Do they agree?** For each hypothesis: how well does the cosine score rank the reviews the LLM said "yes" to above the ones it said "no" to (AUROC; 0.5 = no agreement, 1 = perfect), and the correlation between the two.

In [10]:
agree = []
for j, h in enumerate(hypotheses):
    y = llm_df[h].values
    agree.append({"hypothesis": h, "LLM prevalence": y.mean(),
                  "agreement AUROC": roc_auc_score(y, cos[:, j]) if 0 < y.mean() < 1 else np.nan,
                  "correlation": pearsonr(y, cos[:, j])[0] if 0 < y.mean() < 1 else np.nan})
agree_df = pd.DataFrame(agree).sort_values("agreement AUROC", ascending=False)
display(agree_df.round(3))
print(f"median agreement AUROC: {agree_df['agreement AUROC'].median():.3f}")

,hypothesis,LLM prevalence,agreement AUROC,correlation
7,mentions prolonged menstrual bleeding,0.186,0.918,0.650
19,"mentions specific weight metrics (e.g., pounds, lbs, or numerical values) related to weight loss progress",0.142,0.913,0.626
5,mentions severe itching and burning from yeast infection treatment,0.076,0.913,0.533
1,"mentions severe psychological symptoms including depression, suicidal thoughts, and panic attacks",0.328,0.878,0.615
16,mentions not getting pregnant,0.254,0.871,0.604
0,mentions a specific dosage in milligrams,0.258,0.858,0.547
11,"mentions specific side effects that include nausea and headaches, along with mood swings and weight changes",0.469,0.848,0.585
18,"mentions experiencing severe side effects such as vision blurriness, lethargy, and dry skin",0.464,0.825,0.547
6,mentions severe nausea and vomiting,0.377,0.825,0.540
4,mentions a specific adverse reaction with multiple symptoms,0.678,0.817,0.509


median agreement AUROC: 0.808


**2. Same conclusions?** Run the same held-out evaluation (logistic regression on all 20 hypotheses, Bonferroni p < 0.005) three ways:
- LLM yes/no annotations (as in the main run)
- cosine scores, binarized: for each hypothesis, the top-scoring reviews are marked "yes", with the same number of "yes" reviews as the LLM gave (so the two differ only in *which* reviews they mark)
- raw cosine scores (no binarization, no LLM information at all)

In [11]:
binarized = {}
for j, h in enumerate(hypotheses):
    k = int(llm_df[h].sum())
    b = np.zeros(len(holdout_texts), dtype=int)
    b[np.argsort(cos[:, j])[::-1][:k]] = 1
    binarized[h] = b
zscored = {h: (cos[:, j] - cos[:, j].mean()) / cos[:, j].std() for j, h in enumerate(hypotheses)}

m_llm, t_llm = score_hypotheses(llm_ann, holdout_labels, classification=True)
m_bin, t_bin = score_hypotheses(binarized, holdout_labels, classification=True)
m_raw, coefs_raw = compute_ols_metrics(hypothesis_annotations=zscored, y_true=holdout_labels, classification=True)
threshold = 0.1 / len(hypotheses)

compare = pd.DataFrame({"hypothesis": hypotheses})
for name, coefs in [("LLM", dict(zip(t_llm.hypothesis, zip(t_llm.regression_coef, t_llm.regression_pval)))),
                    ("embedding (binarized)", dict(zip(t_bin.hypothesis, zip(t_bin.regression_coef, t_bin.regression_pval)))),
                    ("embedding (raw)", coefs_raw)]:
    compare[f"{name} coef"] = [coefs[h][0] for h in hypotheses]
    compare[f"{name} significant"] = [coefs[h][1] < threshold for h in hypotheses]
display(compare.round(3))

summary = pd.DataFrame([
    {"annotation method": "LLM (Qwen3-8B)", "held-out AUROC": m_llm["auroc"], "significant": m_llm["Significant"][0], "time (s)": llm_seconds},
    {"annotation method": "embedding, binarized", "held-out AUROC": m_bin["auroc"], "significant": m_bin["Significant"][0], "time (s)": emb_seconds},
    {"annotation method": "embedding, raw cosine", "held-out AUROC": m_raw["auroc"], "significant": sum(p < threshold for _, p in coefs_raw.values()), "time (s)": emb_seconds},
])
display(summary.round(3))
sig = {c: set(compare.hypothesis[compare[f"{c} significant"]]) for c in ["LLM", "embedding (binarized)", "embedding (raw)"]}
print("significant for both LLM and binarized embedding:", len(sig["LLM"] & sig["embedding (binarized)"]), "of", len(sig["LLM"]), "LLM-significant")
print("coefficient signs agree (LLM vs binarized):", int((np.sign(compare["LLM coef"]) == np.sign(compare["embedding (binarized) coef"])).sum()), "/ 20")

Optimization terminated successfully.
         Current function value: 0.379665
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.537225
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.401752
         Iterations 7


,hypothesis,LLM coef,LLM significant,embedding (binarized) coef,embedding (binarized) significant,embedding (raw) coef,embedding (raw) significant
0,mentions a specific dosage in milligrams,-0.033,False,0.105,False,-0.359,True
1,"mentions severe psychological symptoms including depression, suicidal thoughts, and panic attacks",-1.025,True,-0.773,True,-0.456,False
2,mentions the medication caused the user to feel happy or energetic,1.899,True,0.700,True,1.191,True
3,mentions the medication not working,-2.116,True,-0.540,False,-1.216,True
4,mentions a specific adverse reaction with multiple symptoms,-0.657,True,-0.558,True,-0.797,True
5,mentions severe itching and burning from yeast infection treatment,-0.474,False,-0.062,False,0.067,False
6,mentions severe nausea and vomiting,-0.218,False,-0.556,False,0.816,True
7,mentions prolonged menstrual bleeding,-0.438,False,-0.170,False,0.091,False
8,mentions anxiety or panic attacks,0.364,False,0.267,False,0.257,False
9,mentions the medication being effective for a specific pain-related condition,2.094,True,0.716,True,0.547,True


,annotation method,held-out AUROC,significant,time (s)
0,LLM (Qwen3-8B),0.911,8,1877.011
1,"embedding, binarized",0.805,9,13.421
2,"embedding, raw cosine",0.900,13,13.421


significant for both LLM and binarized embedding: 6 of 8 LLM-significant
coefficient signs agree (LLM vs binarized): 17 / 20


In [12]:
agree_df.to_csv(f"{OUT}/followup_agreement.csv", index=False)
compare.to_csv(f"{OUT}/followup_llm_vs_embedding.csv", index=False)
summary.to_csv(f"{OUT}/followup_summary.csv", index=False)
for f in ["followup_agreement.csv", "followup_llm_vs_embedding.csv", "followup_summary.csv"]:
    files.download(f"{OUT}/{f}")

(saved /content/results/followup_agreement.csv)
(saved /content/results/followup_llm_vs_embedding.csv)
(saved /content/results/followup_summary.csv)
